In [1]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")
from IPython.display import display, Markdown

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
os.environ["XAI_API_KEY"] = os.getenv("XAI_API_KEY")


In [2]:
from langchain.tools import tool

@tool
def get_weather(city:str):
    """Get the weather for a city"""
    return f"the weather in the ${city} is sunny"


In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="google_genai:gemini-3.1-flash-lite")

model_with_tools = model.bind_tools([get_weather])


In [10]:
response = model_with_tools.invoke("What is the weather in Boston")
tools = response.tool_calls

for tool in tools:
    print(f"name: {tool["name"]}")
    print(f"Args: {tool["args"]}")


name: get_weather
Args: {'city': 'Boston'}


In [13]:
#Tools Loop Exchange

messages = [{"role":"user", "content":"What is the weather in Boston"}]
ai_message = model_with_tools.invoke(messages)
messages.append(ai_message)

for tool_call in ai_message.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(final_response)

content=[{'type': 'text', 'text': 'The weather in Boston is currently sunny.', 'extras': {'signature': 'EjQKMgEMOdbHcfPU3Hhvw0hRExNAGsbOYwPy/VuCgyKCVo+vmZJosOF8v0EcarHIe6BG/Y5z'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019e5b9b-1f12-7300-90e3-6b994bd80a61-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 87, 'output_tokens': 8, 'total_tokens': 95, 'input_token_details': {'cache_read': 0}}


The Above was the manual way of Langraph where 
1. Make request to LLM but LLm sees it doesn't ahve enough real time data so it looks in the tools and find out it has get weather tool which is to be called
2. Sincle LLM can not directly execute My python code i look into the tool_calls which LLM made and manuall made those call in loops
3. at last i provided with all the context that i get with tools and created a final request and received a final response

What messages looks like:

Python
[
    {"role": "user", "content": "What is the weather in Boston"}
]

Step 1: After the First Model Invoke
Python
ai_message = model_with_tools.invoke(messages)
messages.append(ai_message)
What messages looks like now:

Python
[
    {"role": "user", "content": "What is the weather in Boston"},
    AIMessage(
        content="", 
        tool_calls=[{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'call_abc123'}]
    )
]


Step 2: Inside the Tool Loop
Python
for tool_call in ai_message.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)
What messages looks like now:

Python
[
    {"role": "user", "content": "What is the weather in Boston"},
    AIMessage(content="", tool_calls=[...]),
    ToolMessage(
        content="Weather in Boston is sunny.", 
        name="get_weather", 
        tool_call_id="call_abc123"
    )
]

Step 3: The Final Return Call
Python
final_response = model_with_tools.invoke(messages)
print(final_response)
When you run this final line, you pass the whole 3-item history list back to the engine. The model opens the history book and reads it like a narrative script:

User asked for Boston's weather.
I requested the get_weather tool.
The system ran it and said it's "sunny."

Because it has all the context it needs, it returns the final response object:

Python
AIMessage(content="The current weather in Boston is sunny.")

That above was Manual way the other is using creat_agent from langchain as it automatically bridges teh flow of python code and LLM model 

The moment you introduce Tools, you are no longer just talking to an LLM; you are spinning up an Application Runtime.

The LLM cannot run your Python code. It is physically impossible for a model hosted on Google's cloud servers to reach down, execute a local Python function like get_weather(city), grab the string, and continue talking.

create_agent isn't just a configuration shortcut—it is actually compiling a complete LangGraph State Machine behind the scenes. This machine wraps around your model and takes care of the infrastructure tasks the LLM can't touch:

Memory State Management: It creates an appending array to hold the history of user questions, LLM tool requests, and raw text answers.

The Infrastructure Coordinator: It hosts the conditional code that watches the LLM's output. If it spots a tool request, it pauses the LLM, finds your Python function, runs it locally, and feeds the text back to the model.

In [17]:
from langchain.agents import create_agent
# --- RUN THIS ONCE AT STARTUP (Builds the infrastructure) ---
my_agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[get_weather],
    system_prompt="You are a helpful assistant."
)

# --- CALL THIS AS MANY TIMES AS YOU WANT (Just like before) ---
res1 = my_agent.invoke({"messages": [{"role": "user", "content": "What's the weather in Boston?"}]})
print(res1["messages"][-1].content)

res2 = my_agent.invoke({"messages": [{"role": "user", "content": "What about Paris?"}]})
print(res2["messages"][-1].content)

[{'type': 'text', 'text': 'The weather in Boston is currently sunny.', 'extras': {'signature': 'EjQKMgEMOdbHVEgxsbtFYR7+wvPFO2BIsUPXi3mZeHiQLi4GdqXPR6zdAPKNjanC26rvPKNa'}}]
[{'type': 'text', 'text': 'The weather in Paris is currently sunny.', 'extras': {'signature': 'EjQKMgEMOdbH0EUBiBXSvOVJI1CdwekzYrYzzihW13beTPCI01eAeFo/YcH+H7TvhQFEGVeO'}}]
